# Problema do Caixeiro Viajante — Formulação MTZ

**Trabalho AV3 — Modelagem em Programação Matemática (Unifor)**

## Enunciado

O Problema do Caixeiro Viajante (PCV), ou *Traveling Salesman Problem* (TSP), consiste em determinar a **rota de menor custo** que permite a um viajante visitar um conjunto de cidades **exatamente uma vez** e **retornar à cidade de origem**.

Dado um dígrafo completo $D = (V, A)$ com $|V| = n$ cidades e custo $c_{ij}$ em cada arco $(i, j) \in A$, o objetivo é encontrar um **ciclo Hamiltoniano de custo mínimo**.

---

## Formulação — PLI com eliminação de subciclos (Miller-Tucker-Zemlin)

### Variáveis de decisão

- $x_{ij} \in \{0, 1\}$: vale 1 se o arco $(i \to j)$ pertence ao ciclo ótimo.
- $u_i \in \mathbb{Z}$, $i = 2, \ldots, n$: variáveis auxiliares de ordem de visita.

### Função objetivo

$$\min \sum_{(i,j) \in A} c_{ij} \cdot x_{ij}$$

### Restrições

**Cada cidade tem exatamente um arco de saída:**
$$\sum_{j \neq i} x_{ij} = 1, \quad \forall\, i \in V$$

**Cada cidade tem exatamente um arco de entrada:**
$$\sum_{i \neq j} x_{ij} = 1, \quad \forall\, j \in V$$

**Eliminação de subciclos (MTZ):**
$$u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall\, i, j \in V \setminus \{1\},\; i \neq j$$

**Limites das variáveis de ordem:**
$$2 \leq u_i \leq n, \quad \forall\, i \in V \setminus \{1\}$$

**Domínio:**
$$x_{ij} \in \{0, 1\}, \quad u_i \in \mathbb{Z}$$

In [1]:
!pip install ortools -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python310\python.exe -m pip install --upgrade pip


In [2]:
from ortools.linear_solver import pywraplp
import pandas as pd

## 1. Leitura dos dados de entrada

In [3]:
# Leitura dos arquivos CSV
df_dados_gerais = pd.read_csv('dados/dados-gerais.csv')
df_dados_arcos = pd.read_csv('dados/dados-arcos.csv')

print('Dados gerais:')
print(df_dados_gerais)
print()
print('Dados dos arcos:')
print(df_dados_arcos)

Dados gerais:
   num_vertices
0             5

Dados dos arcos:
    origem  destino  custo
0        1        2     10
1        1        3     15
2        1        4     20
3        1        5     25
4        2        1     12
5        2        3     35
6        2        4      8
7        2        5     18
8        3        1     14
9        3        2     30
10       3        4     22
11       3        5      9
12       4        1     19
13       4        2      7
14       4        3     21
15       4        5     16
16       5        1     24
17       5        2     17
18       5        3     11
19       5        4     15


In [4]:
# Extrair o numero de vertices
num_vertices = int(df_dados_gerais['num_vertices'][0])

# Criar a lista de vertices (1, 2, ..., n)
vertices = []
for i in range(1, num_vertices + 1):
    vertices.append(i)

# Extrair os arcos do dataframe
arcos = []
for row in df_dados_arcos.itertuples():
    arcos.append((row.origem, row.destino, row.custo))

print(f'Numero de vertices: {num_vertices}')
print(f'Vertices: {vertices}')
print(f'Numero de arcos: {len(arcos)}')
print(f'Arcos esperados (grafo completo): {num_vertices * (num_vertices - 1)}')

Numero de vertices: 5
Vertices: [1, 2, 3, 4, 5]
Numero de arcos: 20
Arcos esperados (grafo completo): 20


## 2. Criação do solver e variáveis de decisão

In [5]:
# Criar o solver SCIP
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

# Variaveis x[i,j]: vale 1 se o arco (i -> j) esta no ciclo otimo (binaria)
x = {}
for a in arcos:
    i, j = a[0], a[1]
    x[(i, j)] = solver.BoolVar(f'x{i}{j}')

# Variaveis u[i]: ordem de visita da cidade i (inteira, 2 <= u[i] <= n)
# A cidade 1 nao precisa de variavel de ordem (e a origem/retorno)
u = {}
for i in vertices:
    if i != 1:
        u[i] = solver.IntVar(2, num_vertices, f'u{i}')

print(f'Variaveis x[i,j] criadas: {len(x)}')
print(f'Variaveis u[i] criadas: {len(u)}')

Variaveis x[i,j] criadas: 20
Variaveis u[i] criadas: 4


## 3. Função objetivo

Minimizar o custo total da rota:
$$\min \sum_{(i,j) \in A} c_{ij} \cdot x_{ij}$$

In [6]:
# Funcao objetivo: minimizar o custo total do ciclo
objetivo = solver.Objective()
for a in arcos:
    i, j, c = a[0], a[1], a[2]
    objetivo.SetCoefficient(x[(i, j)], c)
objetivo.SetMinimization()

## 4. Restrições

### 4.1 Restrição de saída
Cada cidade tem exatamente 1 arco saindo:
$$\sum_{j \neq i} x_{ij} = 1, \quad \forall\, i \in V$$

In [7]:
# Restricao de saida: cada cidade tem exatamente 1 arco saindo
for v in vertices:
    restricao = solver.Constraint(1, 1, f'saida_{v}')
    for a in arcos:
        if a[0] == v:
            i, j = a[0], a[1]
            restricao.SetCoefficient(x[(i, j)], 1)

### 4.2 Restrição de entrada
Cada cidade tem exatamente 1 arco entrando:
$$\sum_{i \neq j} x_{ij} = 1, \quad \forall\, j \in V$$

In [8]:
# Restricao de entrada: cada cidade tem exatamente 1 arco entrando
for v in vertices:
    restricao = solver.Constraint(1, 1, f'entrada_{v}')
    for a in arcos:
        if a[1] == v:
            i, j = a[0], a[1]
            restricao.SetCoefficient(x[(i, j)], 1)

### 4.3 Eliminação de subciclos (MTZ)
$$u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall\, i, j \in V \setminus \{1\},\; i \neq j$$

In [9]:
# Restricoes MTZ: eliminacao de subciclos
# u[i] - u[j] + n * x[i,j] <= n - 1, para todo i,j != 1
for a in arcos:
    i, j = a[0], a[1]
    if i != 1 and j != 1:
        restricao = solver.Constraint(-infinity, num_vertices - 1, f'mtz_{i}_{j}')
        restricao.SetCoefficient(u[i], 1)
        restricao.SetCoefficient(u[j], -1)
        restricao.SetCoefficient(x[(i, j)], num_vertices)

print(f'Total de restricoes: {solver.NumConstraints()}')

Total de restricoes: 22


## 5. Modelo de Programação Linear Inteira (formato LP)

In [10]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 22
\   Variables        : 24
\     Binary         : 20
\     Integer        : 4
\     Continuous     : 0
Minimize
 Obj: +10 x12 +15 x13 +20 x14 +25 x15 +12 x21 +35 x23 +8 x24 +18 x25 +14 x31 +30 x32 +22 x34 +9 x35 +19 x41 +7 x42 +21 x43 +16 x45 +24 x51 +17 x52 +11 x53 +15 x54 
Subject to
 saida_1: +1 x12 +1 x13 +1 x14 +1 x15  = 1
 saida_2: +1 x21 +1 x23 +1 x24 +1 x25  = 1
 saida_3: +1 x31 +1 x32 +1 x34 +1 x35  = 1
 saida_4: +1 x41 +1 x42 +1 x43 +1 x45  = 1
 saida_5: +1 x51 +1 x52 +1 x53 +1 x54  = 1
 entrada_1: +1 x21 +1 x31 +1 x41 +1 x51  = 1
 entrada_2: +1 x12 +1 x32 +1 x42 +1 x52  = 1
 entrada_3: +1 x13 +1 x23 +1 x43 +1 x53  = 1
 entrada_4: +1 x14 +1 x24 +1 x34 +1 x54  = 1
 entrada_5: +1 x15 +1 x25 +1 x35 +1 x45  = 1
 mtz_2_3: +5 x23 +1 u2 -1 u3  <= 4
 mtz_2_4: +5 x24 +1 u2 -1 u4  <= 4
 mtz_2_5: +5 x25 +1 u2 -1 u5  <= 4
 mtz_3_2: +5 x32 -1 u2 +1 u3  <= 4
 mtz_3_4: +5 x34 +1 

## 7. [Modificação Temporária - Q1] Encontrar uma Rota Alternativa

Adicionamos uma restrição para proibir os arcos da rota ótima atual, forçando o solver a encontrar uma rota alternativa (segunda melhor rota ou outra com o mesmo custo ótimo).

**Como usar:**
1. Execute as células de 1 a 5 normalmente.
2. Execute a célula **6. Resolução e solução ótima** abaixo para achar a primeira rota.
3. Execute esta célula (Passo 7) para registrar o corte de rota.
4. Volte e **reexecute** a célula do **Passo 6** para resolver novamente com o corte e ver a nova rota!

In [13]:
# [Q1] Adicionar restrição de corte e printar decisões
c = solver.Constraint(-infinity, num_vertices - 1)
for (i, j), var in x.items():
    if var.solution_value() > 0.5:
        print(f"Adicionando x[{i},{j}] ao corte")
        c.SetCoefficient(var, 1)

## 6. Resolução e solução ótima

In [ ]:
# Resolver o modelo
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    custo_total = int(round(objetivo.Value()))

    # Identificar os arcos utilizados na rota otima
    arcos_utilizados = []
    for a in arcos:
        i, j, c = a[0], a[1], a[2]
        if x[(i, j)].solution_value() > 0.5:
            arcos_utilizados.append((i, j, c))

    # Reconstruir a rota a partir dos arcos
    proximo = {}
    for i, j, c in arcos_utilizados:
        proximo[i] = j

    rota = [1]
    atual = 1
    for passo in range(num_vertices):
        atual = proximo[atual]
        rota.append(atual)

    # Exibir resultados
    print('Solucao otima encontrada.')
    print()
    print(f'Rota: {" -> ".join(str(v) for v in rota)}')
    print(f'Custo total: {custo_total}')
    print()
    print('Arcos utilizados:')
    for i, j, c in arcos_utilizados:
        print(f'  {i} -> {j}  (custo {c})')
    print()

    # Exibir ordem de visita (variaveis u)
    print('Ordem de visita (variaveis u):')
    print(f'  u[1] = 1  (origem)')
    for i in vertices:
        if i != 1:
            print(f'  u[{i}] = {int(round(u[i].solution_value()))}')
else:
    print('O problema nao tem solucao otima.')

Solucao otima encontrada.

Rota: 1 -> 3 -> 5 -> 4 -> 2 -> 1
Custo total: 58

Arcos utilizados:
  1 -> 3  (custo 15)
  2 -> 1  (custo 12)
  3 -> 5  (custo 9)
  4 -> 2  (custo 7)
  5 -> 4  (custo 15)

Ordem de visita (variaveis u):
  u[1] = 1  (origem)
  u[2] = 5
  u[3] = 2
  u[4] = 4
  u[5] = 3


: 